In [1]:
# Install websocket client library (if needed)
import sys
!{sys.executable} -m pip install -q websocket-client

## Example 1: Combined stream (trade + book ticker)

This connects to a combined stream and prints the first 10 messages.

In [ ]:
import json
from websocket import WebSocketApp

STREAMS = ["btcusdt@trade", "btcusdt@bookTicker"]
URL = "wss://stream.binance.us:9443/stream?streams=" + "/".join(STREAMS)

messages_to_print = 10
state = {"count": 0}

def on_message(ws, message):
    data = json.loads(message)
    state["count"] += 1
    print(data)
    if state["count"] >= messages_to_print:
        ws.close()

def on_error(ws, error):
    print("error:", error)

def on_close(ws, status_code, msg):
    print("closed:", status_code, msg)

def on_open(ws):
    print("connected to", URL)

ws = WebSocketApp(
    URL,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
)

ws.run_forever(ping_interval=60, ping_timeout=10)

connected to wss://stream.binance.us:9443/stream?streams=btcusdt@trade/btcusdt@bookTicker
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520872, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.00006000', 'a': '68125.27000000', 'A': '0.01467000'}}
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520876, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.00006000', 'a': '68125.26000000', 'A': '0.00420000'}}
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520872, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.00006000', 'a': '68125.27000000', 'A': '0.01467000'}}
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520876, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.00006000', 'a': '68125.26000000', 'A': '0.00420000'}}
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520878, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.00006000', 'a': '68124.86000000', 'A': '0.01467000'}}
{'stream': 'btcusdt@bookTicker', 'data': {'u': 3081520880, 's': 'BTCUSDT', 'b': '68105.28000000', 'B': '0.0000

False

## Example 2: Subscribe via `/ws` and SUBSCRIBE message

This uses the raw `/ws` endpoint and sends a `SUBSCRIBE` payload.
You can add more streams to the `params` array.

## Q&A: Streams vs WebSocket API

**Q1: What's the difference between `/ws` and the combined stream URL?**
- `/stream?streams=...`: combined streams in the URL; messages are wrapped as `{ "stream": "<name>", "data": ... }`.
- `/ws/<stream>` (raw): Connect to one stream directly; messages are the raw payload without the stream wrapper.
- `/ws` + `SUBSCRIBE`: connect to `/ws` and subscribe dynamically; messages are the raw stream payloads (no wrapper).

**Q2: Which one is faster?**
- Neither is meaningfully faster; update cadence is the same. Choose based on convenience (wrapper vs dynamic subscriptions).

**Q3: What's the difference between WebSocket Streams and WebSocket API?**
- **Streams** (`wss://stream.binance.us:9443`): push market data subscriptions like `btcusdt@trade`.
- **WebSocket API** (`wss://ws-api.binance.us:443/ws-api/v3`): request/response for exchange info and authenticated actions (orders/account).

Use Streams for real-time data feeds; use WebSocket API for commands and authenticated account actions.

## Managing a Local Order Book (BNBBTC)

The following example follows Binance.US guidance:
1. Open a depth stream (`/ws/bnbbtc@depth`).
2. Buffer events received from the stream.
3. Fetch a depth snapshot from `https://www.binance.us/api/v1/depth?symbol=BNBBTC&limit=1000`.
4. Drop events where `u <= lastUpdateId`.
5. First processed event must satisfy `U <= lastUpdateId+1 <= u`.
6. Then ensure each new event has `U == previous_u + 1`.
7. Apply absolute quantities; remove price level if quantity is `0`.
8. Removing a non-existent level is normal.

In [6]:
import json
import threading
import time
from collections import deque
import requests
from websocket import WebSocketApp

DEPTH_STREAM_URL = "wss://stream.binance.us:9443/ws/bnbbtc@depth"
SNAPSHOT_URL = "https://www.binance.us/api/v1/depth?symbol=BNBBTC&limit=1000"

MAX_EVENTS = 200
MAX_WAIT_SECONDS = 20
event_buffer = deque()
buffer_lock = threading.Lock()
stop_event = threading.Event()

def on_message(ws, message):
    event = json.loads(message)
    with buffer_lock:
        event_buffer.append(event)
    if len(event_buffer) >= MAX_EVENTS:
        stop_event.set()
        ws.close()

def on_error(ws, error):
    print("error:", error)
    stop_event.set()

def on_close(ws, status_code, msg):
    print("closed:", status_code, msg)

def start_stream():
    ws = WebSocketApp(
        DEPTH_STREAM_URL,
        on_message=on_message,
        on_error=on_error,
        on_close=on_close,
    )
    ws.run_forever(ping_interval=60, ping_timeout=10)

stream_thread = threading.Thread(target=start_stream, daemon=True)
stream_thread.start()

# 1) Buffer some events before snapshot
time.sleep(1.0)

# 2) Get depth snapshot
snapshot = requests.get(SNAPSHOT_URL, timeout=10).json()
last_update_id = snapshot["lastUpdateId"]

bids = {price: qty for price, qty in snapshot["bids"]}
asks = {price: qty for price, qty in snapshot["asks"]}

def apply_side(side_dict, updates):
    for price, qty in updates:
        if qty == "0" or qty == "0.00000000":
            side_dict.pop(price, None)
        else:
            side_dict[price] = qty

# 3) Process buffered events in order
processed = 0
prev_u = None
initialized = False
start_time = time.monotonic()

while True:
    if stop_event.is_set():
        break
    if time.monotonic() - start_time > MAX_WAIT_SECONDS:
        print("timeout waiting for events")
        break
    with buffer_lock:
        event = event_buffer.popleft() if event_buffer else None
    if event is None:
        time.sleep(0.01)
        continue

    U = event.get("U")
    u = event.get("u")
    if U is None or u is None:
        continue

    # Drop events that are too old
    if u <= last_update_id:
        continue

    if not initialized:
        if not (U <= last_update_id + 1 <= u):
            continue
        initialized = True
    else:
        if prev_u is not None and U != prev_u + 1:
            raise RuntimeError(f"Depth update gap: expected U={prev_u+1}, got U={U}")

    apply_side(bids, event.get("b", []))
    apply_side(asks, event.get("a", []))
    prev_u = u
    processed += 1

    if processed >= MAX_EVENTS:
        break

# Print best bid/ask after processing
best_bid = max(bids.items(), key=lambda x: float(x[0])) if bids else (None, None)
best_ask = min(asks.items(), key=lambda x: float(x[0])) if asks else (None, None)
print("processed events:", processed)
print("best bid:", best_bid, "best ask:", best_ask)

timeout waiting for events
processed events: 0
best bid: ('0.00914700', '3.03000000') best ask: ('0.00928300', '3.00000000')


In [4]:
import json
from websocket import WebSocketApp

URL = "wss://stream.binance.us:9443/ws"
streams = ["btcusdt@trade", "btcusdt@kline_1m"]

messages_to_print = 10
state = {"count": 0}

def on_open(ws):
    subscribe_msg = {"method": "SUBSCRIBE", "params": streams, "id": 1}
    ws.send(json.dumps(subscribe_msg))
    print("subscribed to", streams)

def on_message(ws, message):
    data = json.loads(message)
    state["count"] += 1
    print(data)
    if state["count"] >= messages_to_print:
        ws.close()

def on_error(ws, error):
    print("error:", error)

def on_close(ws, status_code, msg):
    print("closed:", status_code, msg)

ws = WebSocketApp(
    URL,
    on_open=on_open,
    on_message=on_message,
    on_error=on_error,
    on_close=on_close,
)

ws.run_forever(ping_interval=60, ping_timeout=10)

subscribed to ['btcusdt@trade', 'btcusdt@kline_1m']
{'result': None, 'id': 1}
{'e': 'kline', 'E': 1772860500024, 's': 'BTCUSDT', 'k': {'t': 1772860440000, 'T': 1772860499999, 's': 'BTCUSDT', 'i': '1m', 'f': -1, 'L': -1, 'o': '68098.81000000', 'c': '68098.81000000', 'h': '68098.81000000', 'l': '68098.81000000', 'v': '0.00000000', 'n': 0, 'x': True, 'q': '0.00000000', 'V': '0.00000000', 'Q': '0.00000000', 'B': '0'}}
{'e': 'kline', 'E': 1772860500024, 's': 'BTCUSDT', 'k': {'t': 1772860440000, 'T': 1772860499999, 's': 'BTCUSDT', 'i': '1m', 'f': -1, 'L': -1, 'o': '68098.81000000', 'c': '68098.81000000', 'h': '68098.81000000', 'l': '68098.81000000', 'v': '0.00000000', 'n': 0, 'x': True, 'q': '0.00000000', 'V': '0.00000000', 'Q': '0.00000000', 'B': '0'}}
{'e': 'kline', 'E': 1772860502585, 's': 'BTCUSDT', 'k': {'t': 1772860500000, 'T': 1772860559999, 's': 'BTCUSDT', 'i': '1m', 'f': -1, 'L': -1, 'o': '68098.81000000', 'c': '68098.81000000', 'h': '68098.81000000', 'l': '68098.81000000', 'v': '0.

True